# tAilorCode — One public task (not yet executed)

**Status: prepared, not validated with a running model.** This notebook is separate from the Kaggle submission ZIP notebook.

Opening or saving the notebook does not start the test. Connecting a Colab runtime can consume paid credits even before running a cell. Agree on the runtime and its displayed consumption before connecting.

This workflow requires an already provisioned Linux x86_64 GPU environment with Python 3.12, the official competition packages, Gemma model files and public task data. Our previous Colab runtime used Python 3.13 and was not ready. This notebook does **not** install the GPU wheelhouse or download the model/data. Compatibility and memory capacity remain to be verified.

Based on the [organizer's starter notebook](https://www.kaggle.com/code/ryanholbrook/getting-started-gemma-4-developer-agent), inspected September 27, 2026. Uses the published subprocess sandbox, without Docker. Use a dedicated temporary environment without personal files or credentials: evaluation executes repository code.


## 1. Explicit runtime decision

Leave this false until runtime consumption has been reviewed and approved. This flag does not control Colab billing or replace disconnecting the runtime.


In [ ]:
RUNTIME_COST_APPROVED = False
if not RUNTIME_COST_APPROVED:
    raise RuntimeError("Stopped. Review and approve runtime/credit consumption before executing these cells.")


## 2. Get the project and set paths

Run only in the approved runtime. No package installation occurs here. Set the two input paths to existing mounted files; do not enter passwords or tokens. A new run folder preserves previous results.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import subprocess
import sys

if not RUNTIME_COST_APPROVED:
    raise RuntimeError("Runtime approval is required")
REPO = Path("/content/tailorcode-smoke-project")
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/AraceliFradejas/tailorcode-gemma-developer-agent.git", str(REPO)], check=True)
print("Project revision:", subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip())
DATA_ROOT = ""  # Directory containing tasks.jsonl, snapshots, wheels, graphs and embeddings.
MODEL_PATH = ""  # Directory containing the specified Gemma model and tokenizer.
TASK_ID = "fastapi_14786"
OUTPUT = Path("/content/tailorcode-runs") / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")


## 3. Check prerequisites without starting Gemma

A failure is a setup blocker, not an agent score. Passing these checks does not guarantee enough GPU memory or a working package installation.


In [ ]:
if not RUNTIME_COST_APPROVED or not DATA_ROOT or not MODEL_PATH:
    raise RuntimeError("Runtime approval and real DATA_ROOT / MODEL_PATH paths are required")
check = subprocess.run([
    sys.executable, str(REPO / "scripts/check_smoke_environment.py"),
    "--data-root", DATA_ROOT, "--model-path", MODEL_PATH, "--task-id", TASK_ID,
], text=True, capture_output=True)
print(check.stdout)
print(check.stderr)
check.check_returncode()


## 4. Run one task — separately opt in

Keep `RUN_AGENT = False` until setup is ready and inference is approved. The runner rechecks prerequisites.

The 5-minute agent budget is **not** a total runtime or spending cap: model startup can take up to 20 minutes, with additional preparation and verification time. This diagnostic budget is shorter than the baseline's 30-minute submission budget; a failed task would not establish baseline performance.

The model server is stopped in a cleanup handler on ordinary completion/failure. **The Colab machine remains connected and can continue consuming credits.** Stop it explicitly after saving outputs. A forceful runtime termination can bypass cleanup.


In [ ]:
RUN_AGENT = False
if not RUNTIME_COST_APPROVED or not RUN_AGENT:
    raise RuntimeError("Inference is disabled. Review environment and consumption before opting in.")
if not DATA_ROOT or not MODEL_PATH:
    raise RuntimeError("Set input paths first")
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
log_path = OUTPUT.parent / (OUTPUT.name + ".log")
print("Output folder:", OUTPUT)
print("Log:", log_path)
with log_path.open("x") as log:
    run = subprocess.run([
        sys.executable, "-u", str(REPO / "scripts/run_public_smoke.py"),
        "--data-root", DATA_ROOT, "--model-path", MODEL_PATH,
        "--task-id", TASK_ID, "--output", str(OUTPUT), "--run-agent",
    ], stdout=log, stderr=subprocess.STDOUT)
print(log_path.read_text()[-12000:])
run.check_returncode()


## 5. Read and preserve the result

`resolved` is the public task verification outcome. `agent.patch` contains generated changes. This is not a Kaggle leaderboard score. Download the output folder and its adjacent log before deleting the runtime; saving the notebook to Drive alone does not preserve runtime files. Do not publish task reference patches or credentials.

Finally use **Runtime → Disconnect and delete runtime**, then verify there are no active sessions. This notebook never submits to Kaggle.


In [ ]:
report_file = OUTPUT / "run.json"
if not report_file.exists():
    print("No run report exists yet; the task has not reached the evaluation runner.")
else:
    print(report_file.read_text())
